# Ultrasonic Signal Classifier Demo

This notebook demonstrates the ultrasonic signal classification system:
- Generates synthetic A-scan signals
- Visualizes sample signals from both classes
- Trains a 1D CNN model
- Evaluates performance with confusion matrix
- Shows predictions on sample signals

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import yaml

from src.signal_generator import UltrasonicSignalGenerator
from src.model import Conv1DClassifier

plt.style.use('seaborn-v0_8-darkgrid')
print("Imports successful!")

## 2. Load Configuration

In [ ]:
# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")

## 3. Generate Synthetic Signals

In [ ]:
# Initialize signal generator
generator = UltrasonicSignalGenerator(
    signal_length=config['signal_length'],
    noise_std=config['noise_std'],
    defect_amplitude=config['defect_amplitude'],
    seed=config['random_seed']
)

# Generate dataset
print("Generating synthetic signals...")
signals, labels = generator.generate_dataset(
    n_defect=config['n_defect_samples'],
    n_no_defect=config['n_no_defect_samples']
)

print(f"Dataset shape: {signals.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Class distribution: {np.bincount(labels)}")

## 4. Visualize Sample Signals

In [ ]:
# Plot sample signals from each class
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Sample Ultrasonic A-Scan Signals', fontsize=16, fontweight='bold')

# No-defect samples
no_defect_indices = np.where(labels == 0)[0][:3]
for i, idx in enumerate(no_defect_indices):
    axes[0, i].plot(signals[idx], linewidth=1.5)
    axes[0, i].set_title(f'No Defect Sample {i+1}', fontweight='bold')
    axes[0, i].set_ylabel('Amplitude')
    axes[0, i].grid(True, alpha=0.3)

# Defect samples
defect_indices = np.where(labels == 1)[0][:3]
for i, idx in enumerate(defect_indices):
    axes[1, i].plot(signals[idx], color='red', linewidth=1.5)
    axes[1, i].set_title(f'Defect Sample {i+1}', fontweight='bold')
    axes[1, i].set_xlabel('Time Sample')
    axes[1, i].set_ylabel('Amplitude')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sample signals visualized!")

## 5. Prepare Data and Create DataLoaders

In [ ]:
# Split data
X_temp, X_test, y_temp, y_test = train_test_split(
    signals, labels,
    test_size=config['test_split'],
    random_state=config['random_seed'],
    stratify=labels
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=config['val_split'],
    random_state=config['random_seed'],
    stratify=y_temp
)

# Convert to torch tensors
X_train = torch.from_numpy(X_train).unsqueeze(1)
X_val = torch.from_numpy(X_val).unsqueeze(1)
X_test = torch.from_numpy(X_test).unsqueeze(1)

y_train = torch.from_numpy(y_train).long()
y_val = torch.from_numpy(y_val).long()
y_test = torch.from_numpy(y_test).long()

# Create dataloaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 6. Create and Train Model

In [ ]:
# Create model
model = Conv1DClassifier(
    input_length=config['signal_length'],
    num_filters=config['num_filters'],
    kernel_size=config['kernel_size'],
    depth=config['depth'],
    dropout_rate=config['dropout_rate'],
    num_classes=2
).to(device)

print(f"Model created with configuration:")
print(model.get_config())
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")
print(f"\nModel:\n{model}")

In [ ]:
# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Training history
train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_val_loss = float('inf')
patience_counter = 0

print("Starting training...")
for epoch in range(config['epochs']):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for signals, labels_batch in train_loader:
        signals, labels_batch = signals.to(device), labels_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels_batch.size(0)
        train_correct += (predicted == labels_batch).sum().item()
    
    train_loss /= len(train_loader)
    train_acc = 100.0 * train_correct / train_total
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for signals, labels_batch in val_loader:
            signals, labels_batch = signals.to(device), labels_batch.to(device)
            
            outputs = model(signals)
            loss = criterion(outputs, labels_batch)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels_batch.size(0)
            val_correct += (predicted == labels_batch).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = 100.0 * val_correct / val_total
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{config['epochs']} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= config['early_stopping_patience']:
            print(f"Early stopping at epoch {epoch + 1}")
            break

print("Training completed!")

## 7. Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(train_losses, label='Training Loss', linewidth=2)
ax1.plot(val_losses, label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(train_accs, label='Training Accuracy', linewidth=2)
ax2.plot(val_accs, label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluate on Test Set

In [ ]:
# Evaluate on test set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for signals, labels_batch in test_loader:
        signals = signals.to(device)
        outputs = model(signals)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_accuracy = np.mean(all_preds == all_labels)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['No Defect', 'Defect']))

## 9. Plot Confusion Matrix

In [ ]:
# Compute and plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Defect', 'Defect'],
            yticklabels=['No Defect', 'Defect'],
            cbar_kws={'label': 'Count'},
            ax=ax)

ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix:\n{cm}")

## 10. Test on Sample Signals

In [ ]:
# Get sample signals for visualization
sample_no_defect = X_test[all_labels == 0][0].unsqueeze(0).to(device)
sample_defect = X_test[all_labels == 1][0].unsqueeze(0).to(device)

# Make predictions
with torch.no_grad():
    pred_no_defect = model(sample_no_defect)
    pred_defect = model(sample_defect)
    
    prob_no_defect = torch.softmax(pred_no_defect, dim=1)[0].cpu().numpy()
    prob_defect = torch.softmax(pred_defect, dim=1)[0].cpu().numpy()

# Plot predictions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# No-defect signal
axes[0, 0].plot(sample_no_defect.cpu().squeeze().numpy(), linewidth=2, color='blue')
axes[0, 0].set_title('No-Defect Sample Signal', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].bar(['No Defect', 'Defect'], prob_no_defect, color=['green', 'red'], alpha=0.7)
axes[0, 1].set_ylabel('Probability')
axes[0, 1].set_title(f'Prediction: No Defect (Confidence: {prob_no_defect[0]:.2%})', fontsize=12, fontweight='bold')
axes[0, 1].set_ylim([0, 1])
for i, v in enumerate(prob_no_defect):
    axes[0, 1].text(i, v + 0.02, f'{v:.2%}', ha='center', fontweight='bold')

# Defect signal
axes[1, 0].plot(sample_defect.cpu().squeeze().numpy(), linewidth=2, color='red')
axes[1, 0].set_title('Defect Sample Signal', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Time Sample')
axes[1, 0].set_ylabel('Amplitude')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].bar(['No Defect', 'Defect'], prob_defect, color=['green', 'red'], alpha=0.7)
axes[1, 1].set_ylabel('Probability')
axes[1, 1].set_title(f'Prediction: Defect (Confidence: {prob_defect[1]:.2%})', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim([0, 1])
axes[1, 1].set_xlabel('Class')
for i, v in enumerate(prob_defect):
    axes[1, 1].text(i, v + 0.02, f'{v:.2%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("Predictions on sample signals completed!")

## Summary

This demo has successfully:
- Generated synthetic ultrasonic A-scan signals with defect and no-defect classes
- Visualized sample signals to understand the difference between classes
- Trained a 1D CNN classifier on the synthetic dataset
- Evaluated the model performance using multiple metrics
- Plotted the confusion matrix and training history
- Made predictions on new signals

### Next Steps:
1. **Run hyperparameter tuning**: Use `python hyperparameter_tuning.py --n-trials 50` to find optimal hyperparameters
2. **Train final model**: Use `python train.py --config config.yaml` to train the final model
3. **Evaluate**: Use `python evaluate.py --model models/best_model.pt` to evaluate on test set
4. **View MLflow results**: Open MLflow UI with `mlflow ui --backend-store-uri file:./logs/mlflow`